# LC 76 — Minimum Window Substring

| Field      | Value                                         |
|------------|-----------------------------------------------|
| Difficulty | Hard                                          |
| Category   | String / Sliding Window                       |
| Pattern    | Variable-size window with frequency matching  |

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Expand the right pointer until
the window contains every character of <code>t</code>.
Then greedily shrink from the left to find the smallest
valid window. Record it, then continue expanding.
Track "formed" characters to know instantly when the window
is valid.
</div>


## Official Problem Statement

Given two strings `s` and `t` of lengths `m` and `n`,
return the **minimum window substring** of `s` such that every
character in `t` (including duplicates) is included in the window.
If there is no such substring, return the empty string `""`.

**Constraints:**
- `m == s.length`, `n == t.length`
- `1 <= m, n <= 10^5`
- `s` and `t` consist of uppercase and lowercase English letters.
- The answer is guaranteed to be unique.


## What This Is Actually Asking

You have a string `s` and a target `t` (like a shopping list).
You need to find the shortest contiguous piece of `s` that
contains every character on the list, with the right counts.
You can grab extra characters — the window just can't miss any.
Return that shortest piece, or empty string if impossible.


## Walk Through an Example by Hand

`s = "ADOBECODEBANC"`, `t = "ABC"`
need = {A:1, B:1, C:1}, required = 3

```
Expand right until formed==3:
  R=0 'A': have={A:1}        formed=1
  R=1 'D': have={A:1,D:1}   formed=1
  R=2 'O': ...               formed=1
  R=3 'B': have={...,B:1}   formed=2
  R=4 'E': ...               formed=2
  R=5 'C': have={...,C:1}   formed=3  ← valid!
  Window "ADOBEC" len=6, best=(0,5)

Shrink from left:
  L=0 'A': remove A, have={A:0} formed=2  ← invalid
  Stop shrinking. Window still at L=1, R=5.

Expand right again:
  R=6 'O': formed=2
  R=7 'D': formed=2
  R=8 'E': formed=2
  R=9 'B': formed=2 (B count 1, already had 0 after shrink)
            have={B:1} formed=2  (need A)
  R=10 'A': have={A:1} formed=3 ← valid!
  Window "DOBECODEBA" L=1..R=10, len=10  not better

Shrink from left:
  L=1 'D': remove, formed stays 3  → window "OBECODEBA" len=9
  L=2 'O': remove, formed stays 3  → "BECODEBA" len=8
  L=3 'B': remove B, formed=2 ← stop.  best still (0,5).

Continue... eventually find "BANC" len=4 → new best.
Answer: "BANC"
```


## The Picture

Window condition: **formed == required** (all t chars covered)

```
s =  A  D  O  B  E  C  O  D  E  B  A  N  C
     0  1  2  3  4  5  6  7  8  9  10 11 12

Phase 1 — EXPAND until window is valid:
     [  A  D  O  B  E  C  ]               valid! record len=6
     L                    R

Phase 2 — SHRINK from left while still valid:
        [  D  O  B  E  C  ]               remove 'A', invalid!
        L                 R               stop shrinking.

Phase 3 — EXPAND again until valid:
        [  D  O  B  E  C  O  D  E  B  A  ]
        L                                R
        valid → record, then shrink again ...

Final best window:
                                [  B  A  N  C  ]  len=4
                                L              R
```

Two frequencies to watch:
- `need[c]`  — how many of char c does t require
- `have[c]`  — how many are in the current window
- `formed`   — count of chars where have[c] >= need[c]


## When To Use This Pattern

- When you see **"minimum window"** or **"shortest substring"
  containing all of X"**, think **expand-then-shrink window**.
- When the window validity depends on **frequency matching**
  (not just presence), think **need dict + formed counter**.
- When shrinking is safe only until a required count drops,
  think **stop-shrink-on-violation, resume-expand**.
- When brute force is O(n^2) or O(n^3), think
  **two-pointer sliding window for O(n)**.


## The Approach

Build a `need` dict from `t` and count how many unique chars
must be satisfied (`required`). Use a `have` dict and a
`formed` counter as you move the right pointer.
When `formed == required`, record the window length if it
is the best so far, then shrink from the left — removing
characters until the window becomes invalid again.
Repeat until the right pointer exhausts `s`.


In [ ]:
from typing import List                      # standard type hints
from collections import Counter, defaultdict # Counter for need dict


In [ ]:
def test_harness(func):
    """Run a fixed suite of tests against func(s, t) -> str."""
    tests = [
        # (s, t, expected)
        ("ADOBECODEBANC", "ABC",  "BANC"),  # classic
        ("a",             "a",    "a"),      # single match
        ("a",             "aa",   ""),       # impossible
        ("aa",            "aa",   "aa"),     # exact match
        ("ab",            "b",    "b"),      # single char target
        ("bba",           "ab",   "ba"),     # end of string
        ("cabwefgewcwaefgcf", "cae", "cwae"), # longer example
    ]
    passed = 0
    for s, t, expected in tests:
        result = func(s, t)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"{status} | s={repr(s):<20} t={repr(t):<6} "
            f"expected={repr(expected):<8} got={repr(result)}"
        )
    print(f"\n{passed}/{len(tests)} tests passed.")


In [ ]:
def min_window(s: str, t: str) -> str:
    """
    Return the minimum window in s that contains all chars of t.
    Return "" if no such window exists.

    Approach:
        Build need dict from t. Track have dict and formed count.
        Expand right; when formed==required, record window and
        shrink from left until invalid. Repeat.

    Time:  O(|s| + |t|)  — each char visited at most twice
    Space: O(|s| + |t|)  — need and have dicts
    """
    pass


# --- debug runs (expected values in comments) ---
print(min_window("ADOBECODEBANC", "ABC"))    # "BANC"
print(min_window("a", "a"))                  # "a"
print(min_window("a", "aa"))                 # ""
print(min_window("bba", "ab"))               # "ba"
print(min_window("cabwefgewcwaefgcf", "cae")) # "cwae"


In [ ]:
# Uncomment and run when solution is ready
# test_harness(min_window)


## Complexity

| Approach              | Time            | Space           |
|-----------------------|-----------------|-----------------|
| Brute force (all subs)| O(n^2 * m)      | O(1)            |
| Sliding window        | O(\|s\| + \|t\|)| O(\|s\| + \|t\|)|


## Real World Connection

In Citi's AWS-based telemetry platform, you might need to find
the shortest time window in a Kinesis stream that contains
at least one occurrence of each critical event type —
e.g., AUTH, TRADE, SETTLE — to confirm a full transaction
cycle completed.
The `need` dict maps to required event types; the `have` dict
tracks what the current rolling window has seen.
This pattern also applies to SLA auditing: given a log of
6,000 endpoints' heartbeats, find the shortest interval in
which every endpoint checked in at least once.
The sliding window brings this from a multi-hour batch scan
to a single O(n) pass over the stream.


> **Simplicity and clarity is Gold.** — Sean's Study Mantra
